# exp_7 · 04 target — only if H1 passed (BAR-ATH, aggregates only)
Three full-data `exp7_robust` fits; tactical ATD with exp_6's recovered ball on accepted hidden graded frames and
`ball_recovered` = 1 there; final measured-view configs; plausibility guard (dead share 0.407 ± 0.030); validator;
rebuild with exp_6's detector retrained in a fresh folder + three fresh fits. **Reads BAR-ATH: never open this
notebook executed.** Results: `results/target_candidates.json`.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp7_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_7')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp7_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_04.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_6 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp7_decision as DE
import exp7_fit as F
import exp7_target as TG
log = make_log('04')
decision = C1.read_json(C.RESULTS / 'decision.json')
DE.require_h1(decision)

In [ ]:
models = {arm: TG.full_model(arm) for arm in F.ARMS}
first = TG.build(decision, C.RECOVERED, TG.CANDIDATES, models, log)
C.write_results(C.RESULTS / 'target_candidates.json', {'built_at_utc': now(), 'first': first,
                                                     'target_fits': TG.target_fit_count()})
print({label: {k: v for k, v in info.items() if k != 'sha256'} for label, info in first['files'].items()})

In [ ]:
rebuild = TG.rebuild(decision, first, log)
C.write_results(C.RESULTS / 'target_candidates.json', {'built_at_utc': now(), 'first': first, 'rebuild': rebuild,
                                                     'target_fits': TG.target_fit_count()})
print('rebuild byte-identical:', rebuild['all_byte_identical'])
final = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / 'guard_final.json')
print('final write guard unchanged:', final['all_unchanged'])